# Project 13 · PubMedBERT 疾病实体识别基线

默认微调微软的 **PubMedBERT abstracts-only**（现名 BiomedBERT）。使用队友笔记本对应的 NCBI Disease 划分和首子词标签对齐规则。训练 3 个 epoch，每轮计算验证集指标，固定使用第 3 轮权重评估测试集，以复现队友原有的评估流程。

队友附件里的测试 F1 为 **0.8528**，模型 ID 为 `bert-base-cased`。这组分数属于通用 BERT。正式比较需在同一流程下得到 PubMedBERT 和 BioBERT 各自的实测成绩。

**运行方法**：在有网络的 Jupyter 或 Colab GPU 环境中从上往下运行。Colab 中可通过「运行时 → 更改运行时类型」选择 GPU。首次安装后若提示重启内核，请重启后从配置单元继续。先运行默认的 PubMedBERT，再将 `MODELS_TO_RUN` 改为 `["BioBERT"]` 并从配置单元往下运行。也可以一次运行两个模型。每次运行产物保存在当前工作目录的 `ner_baseline_runs/`。Colab 会话结束前请下载或持久保存该目录。

[PubMedBERT 模型卡](https://huggingface.co/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract) · [BioBERT 模型卡](https://huggingface.co/dmis-lab/biobert-base-cased-v1.2) · [NCBI Disease 数据卡](https://huggingface.co/datasets/ncbi/ncbi_disease)

In [ ]:
%pip install "torch>=2.6,<3" "transformers==4.57.1" "datasets==3.6.0" "accelerate>=1.1,<2" "seqeval==1.2.2"

In [ ]:
import csv
import gc
import hashlib
import json
import tempfile
from pathlib import Path
from urllib.request import urlopen

import datasets
import numpy as np
import torch
import transformers
from datasets import Dataset, DatasetDict
from IPython.display import Markdown, display
from seqeval.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (
    AutoModelForTokenClassification, AutoTokenizer, DataCollatorForTokenClassification,
    Trainer, TrainingArguments, set_seed,
)

MODELS_TO_RUN = ["PubMedBERT"]
MODEL_IDS = {
    "PubMedBERT": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract",
    "BioBERT": "dmis-lab/biobert-base-cased-v1.2",
}
MODEL_REVISIONS = {
    "PubMedBERT": "d673b8835373c6fa116d6d8006b33d48734e305d",
    "BioBERT": "67c9c25b46986521ca33df05d8540da1210b3256",
}
OUTPUT_ROOT = Path("ner_baseline_runs")
SEED = 42
MAX_LENGTH = 512
LABELS = ["O", "B-Disease", "I-Disease"]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
print("CUDA:", torch.cuda.is_available(), "| PyTorch:", torch.__version__)

## 1. 载入数据

直接读取 Hugging Face 加载脚本使用的 CoNLL 数据文件，沿用其分句和标签定义，包括每个文件末尾由原加载脚本产生的一条空记录。检查三个划分的样本数，并记录原始文件的 SHA-256 以核对后续实验。

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/spyysalo/ncbi-disease/master/conll/"
SPLIT_FILES = {"train": "train.tsv", "validation": "devel.tsv", "test": "test.tsv"}
EXPECTED_SIZES = {"train": 5433, "validation": 924, "test": 941}

def parse_conll(content):
    sentences = []
    tokens, tags = [], []
    for line in content.splitlines():
        if not line.strip():
            if tokens:
                sentences.append({"tokens": tokens, "ner_tags": tags})
                tokens, tags = [], []
            continue
        token, tag = line.split("\t")
        tokens.append(token)
        tags.append(LABEL_TO_ID[tag.strip()])
    sentences.append({"tokens": tokens, "ner_tags": tags})
    return sentences

dataset = DatasetDict()
data_hashes = {}
for split, filename in SPLIT_FILES.items():
    with urlopen(DATA_URL + filename, timeout=60) as response:
        raw = response.read()
    data_hashes[split] = hashlib.sha256(raw).hexdigest()
    rows = parse_conll(raw.decode("utf-8"))
    assert len(rows) == EXPECTED_SIZES[split], (split, len(rows))
    dataset[split] = Dataset.from_list(rows)
print({split: len(rows) for split, rows in dataset.items()})

## 2. 对齐标签与定义指标

每个原始词只用首个子词参与训练与评分。其余子词、特殊符号和补齐位置用 `-100` 忽略。Precision、Recall 和 F1 沿用队友的 seqeval 默认实体级规则，Accuracy 是词级标签准确率。每个模型使用自己的分词器，并检查分词后是否覆盖全部原始词。

In [ ]:
def tokenize_and_align_labels(examples, tokenizer):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=MAX_LENGTH,
        is_split_into_words=True
    )

    all_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        label_ids = []
        previous_word_id = None

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(labels[word_id])
            else:
                label_ids.append(-100)

            previous_word_id = word_id

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels

    return tokenized_inputs

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predicted_ids = np.argmax(logits, axis=-1)
    predictions, references = [], []
    for pred_row, label_row in zip(predicted_ids, labels):
        kept = label_row != -100
        predictions.append([LABELS[index] for index in pred_row[kept]])
        references.append([LABELS[index] for index in label_row[kept]])
    return {
        "precision": precision_score(references, predictions, zero_division=0),
        "recall": recall_score(references, predictions, zero_division=0),
        "f1": f1_score(references, predictions, zero_division=0),
        "accuracy": accuracy_score(references, predictions),
    }

## 3. 训练、验证与测试

沿用学习率 2e-5、每设备 batch size 8、3 个 epoch 和 weight decay 0.01，显式固定随机种子 42。支持 BF16 的 CUDA 显卡使用混合精度，两个模型采用相同设置。训练日志含逐轮验证结果。每个模型保存最终权重、分词器、训练配置及指标；不同运行使用独立目录。为节省磁盘，只保存最终模型。

In [ ]:
def run_baseline(name):
    model_id = MODEL_IDS[name]
    revision = MODEL_REVISIONS[name]
    set_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, revision=revision, use_fast=True, do_lower_case=(name == "PubMedBERT")
    )
    encoded = dataset.map(
        tokenize_and_align_labels,
        batched=True,
        fn_kwargs={"tokenizer": tokenizer},
        remove_columns=dataset["train"].column_names,
    )
    for split, rows in encoded.items():
        for index, (aligned, original_tags) in enumerate(zip(rows["labels"], dataset[split]["ner_tags"])):
            kept = [tag for tag in aligned if tag != -100]
            if kept != original_tags:
                raise ValueError(f"{name}: {split} 第 {index} 条出现标签丢失或错位")

    model = AutoModelForTokenClassification.from_pretrained(
        model_id, revision=revision, num_labels=len(LABELS), label2id=LABEL_TO_ID,
        id2label=dict(enumerate(LABELS)),
    )
    model_root = OUTPUT_ROOT / name.lower()
    model_root.mkdir(parents=True, exist_ok=True)
    run_dir = Path(tempfile.mkdtemp(prefix="run-", dir=model_root))
    args = TrainingArguments(
        output_dir=str(run_dir),
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        seed=SEED,
        data_seed=SEED,
        optim="adamw_torch",
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=encoded["train"],
        eval_dataset=encoded["validation"],
        processing_class=tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )
    trainer.train()
    test = trainer.evaluate(encoded["test"], metric_key_prefix="test")
    trainer.save_model(str(run_dir / "model"))
    trainer.state.save_to_json(str(run_dir / "trainer_state.json"))
    result = {
        "model": name,
        "checkpoint": model_id,
        "model_revision": revision,
        "seed": SEED,
        "max_length": MAX_LENGTH,
        "data_sha256": data_hashes,
        "versions": {"torch": str(torch.__version__), "datasets": datasets.__version__,
                     "transformers": transformers.__version__},
        "training_args": args.to_dict(),
        "test": test,
    }
    (run_dir / "results.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(f"{name} 已保存到 {run_dir.resolve()}")

for model_name in MODELS_TO_RUN:
    run_baseline(model_name)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 4. 生成测试集对比表

读取每个模型最近一次与当前数据文件匹配的完整运行。表内为 0–1 小数。通用 BERT 的参考分数来自队友附件。BioBERT 和 PubMedBERT 都有实测结果时，表格会显示 F1 相差多少个百分点。

In [ ]:
columns = ["model", "precision", "recall", "f1", "accuracy"]
rows = [{
    "model": "BERT-base-cased（队友附件）",
    "precision": 0.826171875,
    "recall": 0.88125,
    "f1": 0.8528225806451613,
    "accuracy": 0.9838755766012165,
}]
for name in ("BioBERT", "PubMedBERT"):
    paths = (OUTPUT_ROOT / name.lower()).glob("run-*/results.json")
    row = {"model": name}
    for path in sorted(paths, key=lambda item: item.stat().st_mtime, reverse=True):
        result = json.loads(path.read_text(encoding="utf-8"))
        if (
            result["data_sha256"] == data_hashes
            and result["checkpoint"] == MODEL_IDS[name]
            and result["model_revision"] == MODEL_REVISIONS[name]
        ):
            row.update({key: result["test"][f"test_{key}"] for key in columns[1:]})
            break
    rows.append(row)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
with (OUTPUT_ROOT / "test_comparison.csv").open("w", encoding="utf-8-sig", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=columns)
    writer.writeheader()
    writer.writerows(rows)

table = ["| Model | Precision | Recall | F1 | Accuracy |", "|---|---:|---:|---:|---:|"]
for row in rows:
    values = [f"{row[key]:.4f}" if key in row else "待训练" for key in columns[1:]]
    table.append("| " + " | ".join([row["model"], *values]) + " |")
display(Markdown("\n".join(table)))
if all("f1" in row for row in rows[1:]):
    print(f"PubMedBERT − BioBERT 测试 F1：{100 * (rows[2]['f1'] - rows[1]['f1']):+.2f} 个百分点")